# PBS Crops — Sindh district-wise data via the portal's JSON API
Replaces the Playwright version. Endpoints (from `pbs_discovery.json`):

| Purpose | Request |
|---|---|
| Province / division / district lists | `GET /Crops/GetSelectList/?item=pv\|dv\|ds&parent=<id>` |
| Crop ids | `GET /Crops/GetCrops` (Maize 1, Rice 2, Sugarcane 3, Wheat 4, Cotton 5) |
| Yearly table | `POST /Crops/GetYearly` with `level`, `area`, `crop` |

Known levels: `11` = national (`area=0`), `12` = province (`area=3` for Sindh). Division/district levels were not in the capture, so section 4 **detects them** by checking that children sum to their parent.

**Run order:** 1 -> 2 -> 3 (probe) -> 4 (detect levels) -> 5 (scrape) -> 6 (validate) -> 7 (export).

## 1. Setup and config

In [ ]:
%pip install -q requests pandas

In [ ]:
import json, logging, re, time
from io import StringIO

import pandas as pd
import requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
log = logging.getLogger("pbs")

BASE        = "https://na.data.gov.pk"
OUT_CSV     = "pbs_crops_1981_2025.csv"
PROVINCE    = "SINDH"
CROP_NAMES  = ["Wheat", "Rice", "Cotton", "Sugarcane", "Maize"]
FY_START, FY_END = 1981, 2024      # start-year of fiscal year; use FY_START = 2009 for 2009-10 onward
DELAY       = 0.25                 # seconds between calls (be polite to a government server)
PROVINCE_LEVEL = 12                # confirmed from the XHR capture
LEVEL_CANDIDATES = [13, 14, 15, 16]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124 Safari/537.36",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": f"{BASE}/Crops/Home",
    "Origin": BASE,
    "Accept": "application/json, text/javascript, */*; q=0.01",
})
session.get(f"{BASE}/Crops/Home", timeout=30)       # pick up any session cookies
print("session ready, cookies:", list(session.cookies.keys()))

## 2. Helpers

In [ ]:
def _request(method, path, retries=4, **kw):
    for attempt in range(1, retries + 1):
        try:
            r = session.request(method, BASE + path, timeout=30, **kw)
            r.raise_for_status()
            time.sleep(DELAY)
            return r
        except requests.RequestException as e:
            if attempt == retries:
                raise
            wait = 2 ** attempt
            log.warning("%s %s failed (%s); retry in %ss", method, path, e, wait)
            time.sleep(wait)

def select_list(item, parent=0):
    """-> [{'id': '17', 'name': 'HYDERABAD DIVISION'}, ...]"""
    r = _request("GET", "/Crops/GetSelectList/", params={"item": item, "parent": parent})
    return r.json()["data"]

def num(x):
    return pd.to_numeric(re.sub(r"[^\d.\-]", "", str(x)), errors="coerce")

def fy_start(fy):
    m = re.match(r"\s*(\d{4})", str(fy))
    return int(m.group(1)) if m else None

COLS = ["Year", "Area", "Production", "Yield"]

def parse_yearly(text):
    """Normalise a GetYearly response (JSON list/dict, or an HTML table) to Year/Area/Production/Yield."""
    try:
        obj = json.loads(text)
    except ValueError:
        obj = None
    if obj is None:                                             # HTML fragment fallback
        tables = pd.read_html(StringIO(text))
        df = tables[0] if tables else pd.DataFrame()
    else:
        rows = obj
        while isinstance(rows, dict):                           # unwrap {"data": ...} style envelopes
            lists = [v for v in rows.values() if isinstance(v, (list, dict))]
            if "data" in rows: rows = rows["data"]
            elif lists:        rows = lists[0]
            else:              rows = []
        df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(columns=COLS)

    if all(isinstance(c, int) for c in df.columns) and df.shape[1] >= 4:   # list-of-lists
        df = df.iloc[:, :4]; df.columns = COLS
    mapping = {}
    for c in df.columns:
        low = str(c).lower()
        tgt = ("Year" if ("year" in low or low in ("fy", "fiscal")) else
               "Production" if "prod" in low else
               "Yield" if "yield" in low else
               "Area" if ("area" in low or "cult" in low) else None)
        if tgt and tgt not in mapping.values():
            mapping[c] = tgt
    df = df.rename(columns=mapping)
    missing = [c for c in COLS if c not in df.columns]
    if missing:
        raise ValueError(f"GetYearly response lacks {missing}. Columns: {list(df.columns)}; "
                         f"head: {df.head(3).to_dict('records')}")
    df = df[COLS].copy()
    for c in ("Area", "Production", "Yield"):
        df[c] = df[c].map(num)
    df["Year"] = df["Year"].astype(str).str.strip()
    df["Year_start"] = df["Year"].map(fy_start)
    return df.dropna(subset=["Year_start"]).reset_index(drop=True)

def yearly_raw(level, area, crop_id):
    return _request("POST", "/Crops/GetYearly",
                    data={"level": level, "area": area, "crop": crop_id}).text

def yearly(level, area, crop_id):
    """Parsed yearly series; empty frame if the server returns nothing/garbage for that level+area."""
    try:
        return parse_yearly(yearly_raw(level, area, crop_id))
    except Exception as e:
        log.debug("yearly(%s,%s,%s) -> %s", level, area, crop_id, e)
        return pd.DataFrame(columns=COLS + ["Year_start"])

## 3. Hierarchy and probe
Prints the Sindh divisions/districts and the **raw** `GetYearly` response for Sindh wheat. If section 3 errors on parsing, paste the raw output here and I'll adjust the key mapping.

In [ ]:
provinces = select_list("pv", 0)
sindh = next(p for p in provinces if p["name"].strip().upper() == PROVINCE)
SINDH_ID = sindh["id"]

divisions = select_list("dv", SINDH_ID)
districts = []                                   # [{'division', 'division_id', 'id', 'name'}]
for dv in divisions:
    for d in select_list("ds", dv["id"]):
        districts.append({"division": dv["name"], "division_id": dv["id"], "id": d["id"], "name": d["name"]})

crop_list = select_list_crops = _request("GET", "/Crops/GetCrops").json()["data"]
CROPS = {}
for name in CROP_NAMES:
    hit = next(c for c in crop_list if c["name"].strip().lower() == name.lower())
    CROPS[name] = hit["id"]

print("Sindh id:", SINDH_ID)
print("Divisions:", [(d["id"], d["name"]) for d in divisions])
print(f"{len(districts)} districts")
display(pd.DataFrame(districts))
print("Crop ids:", CROPS)

In [ ]:
raw = yearly_raw(PROVINCE_LEVEL, SINDH_ID, CROPS["Wheat"])
print("RAW (first 1200 chars):\n", raw[:1200], "\n")
prov_wheat = parse_yearly(raw)
display(prov_wheat.head(), prov_wheat.tail())
print(prov_wheat.shape, "| years:", prov_wheat.Year.iloc[0], "->", prov_wheat.Year.iloc[-1])

## 4. Detect the division and district `level` codes
Picks the level whose children add up to the parent (Sindh = sum of divisions; a division = sum of its districts), using wheat production in the latest year with data.

In [ ]:
def latest(df, col="Production"):
    d = df.dropna(subset=[col]); d = d[d[col] > 0]
    return d.sort_values("Year_start").iloc[-1] if len(d) else None

def value_at(df, year_start, col="Production"):
    m = df[df.Year_start == year_start][col].dropna()
    return float(m.iloc[0]) if len(m) else 0.0

def children_error(parent_total, year_start, level, child_ids, crop_id):
    s = sum(value_at(yearly(level, cid, crop_id), year_start) for cid in child_ids)
    return abs(s - parent_total) / parent_total, s

wheat = CROPS["Wheat"]
ref = latest(prov_wheat)
Y, sindh_total = int(ref.Year_start), float(ref.Production)
print(f"Reference: Sindh wheat production {Y}-{(Y+1)%100:02d} = {sindh_total:,.0f}")

# division level
div_ids = [d["id"] for d in divisions]
res = {L: children_error(sindh_total, Y, L, div_ids, wheat) for L in LEVEL_CANDIDATES}
for L, (err, s) in res.items(): print(f"  level {L} as DIVISION: sum={s:,.0f}  rel.err={err:.2%}")
DIV_LEVEL = min(res, key=lambda L: res[L][0])
assert res[DIV_LEVEL][0] < 0.10, "No candidate level reproduces the Sindh total - paste the output above."

# district level: test on the division with the most districts
by_div = pd.DataFrame(districts).groupby("division_id")["id"].apply(list)
test_div = max(by_div.index, key=lambda k: len(by_div[k]))
div_total = value_at(yearly(DIV_LEVEL, test_div, wheat), Y)
res2 = {L: children_error(div_total, Y, L, by_div[test_div], wheat)
        for L in LEVEL_CANDIDATES if L != DIV_LEVEL}
for L, (err, s) in res2.items(): print(f"  level {L} as DISTRICT: sum={s:,.0f}  rel.err={err:.2%}")
DIST_LEVEL = min(res2, key=lambda L: res2[L][0])
assert res2[DIST_LEVEL][0] < 0.10, "No candidate level reproduces the division total - paste the output above."
print(f"\nDetected levels -> division={DIV_LEVEL}, district={DIST_LEVEL}")

## 5. Full scrape (districts × crops)

In [ ]:
def scrape():
    frames = []
    for i, d in enumerate(districts, 1):
        for crop, cid in CROPS.items():
            try:
                df = yearly(DIST_LEVEL, d["id"], cid)
            except Exception as e:
                log.error("FAILED %s / %s: %s", d["name"], crop, e); continue
            if df.empty:
                log.warning("no data: %s / %s", d["name"], crop); continue
            df = df.assign(Division=d["division"], District=d["name"], District_ID=d["id"], Crop=crop)
            frames.append(df)
        log.info("[%d/%d] %s done", i, len(districts), d["name"])
        if frames:
            pd.concat(frames).to_csv(OUT_CSV + ".partial", index=False)       # checkpoint
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

raw_df = scrape()
raw_df.shape

## 6. Clean and validate

In [ ]:
df = raw_df.copy()
df = df[df.Year_start.between(FY_START, FY_END)]
df.insert(0, "Province", PROVINCE.title())
df["Division"] = df["Division"].str.replace(r"\s+DIVISION$", "", regex=True).str.title()
df["District"] = df["District"].str.replace(r"\s+DISTRICT$", "", regex=True).str.title()
df = (df.drop_duplicates(["District_ID", "Crop", "Year"])
        .sort_values(["District", "Crop", "Year_start"]).reset_index(drop=True))
df = df[["Province", "Division", "District", "District_ID", "Crop", "Year", "Year_start",
         "Area", "Production", "Yield"]]

coverage = df.pivot_table(index="District", columns="Crop", values="Year", aggfunc="count", fill_value=0)
display(coverage)
print("Null counts:\n", df[["Area", "Production", "Yield"]].isna().sum())

# Reconcile district sums with the provincial total (latest year, per crop)
rows = []
for crop, cid in CROPS.items():
    p = latest(yearly(PROVINCE_LEVEL, SINDH_ID, cid))
    if p is None: continue
    y = int(p.Year_start)
    s = df[(df.Crop == crop) & (df.Year_start == y)].Production.sum()
    rows.append({"Crop": crop, "Year": p.Year, "Province": p.Production, "Sum of districts": s,
                 "Diff %": round(100 * (s - p.Production) / p.Production, 2)})
display(pd.DataFrame(rows))

## 7. Export

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"Saved {len(df):,} rows -> {OUT_CSV}  ({df.District.nunique()} districts, {df.Crop.nunique()} crops, "
      f"{df.Year_start.min()}-{df.Year_start.max()+1})")